In [1]:
!pip install -U gradio

In [ ]:
import gradio as gr
import torch
from ALL_WSD_Functions import (
    predict_sense, get_synsets, lem, clean, tokenize, find_target, window,
    ctx_features, gloss_features, encode, model, SYN_ROW, GLOSS_TABLE, CFG
)

def wsd_ui(sentence, target, lemma):
    if not sentence.strip() or not target.strip():
        return "Please enter both a Kannada sentence and a target word."
    
    try:
        # Resolve target indices and predictions safely using backend inference functions
        target_clean = clean(target)
        lemma_clean = clean(lemma) if lemma.strip() else lem(target_clean)[0]
        tokens = tokenize(sentence)
        i, method = find_target(tokens, target_clean, lemma_clean)
        
        if i is None:
            return f"**Could not predict:** '{target_clean}' (lemma '{lemma_clean}') not found in sentence tokens."
            
        left, right = window(tokens, i, CFG["window"])
        cands = get_synsets(lemma_clean)
        if not cands:
            return f"**Could not predict:** '{lemma_clean}' is not present in the Kannada IndoWordNet baseline."
            
        # Run inference via the backend functions
        res = predict_sense(sentence, target, lemma.strip() or None, top_k=3)
    except Exception as e:
        return f"**Internal Engine Error:** {str(e)}"
        
    # Formatting output structure dynamically
    out = [f"**Target:** {res['target']} | **Lemma:** {res['lemma']}",
           f"**Context:** {' '.join(res['left'])} **[{res['target']}]** {' '.join(res['right'])}"]
    if res["note"]: 
        out.append(f"_{res['note']}_")
    
    out.append("\n### Predicted Senses")
    for r in res["ranked"]:
        out.append(f"- **{r['prob']:.1%}** [{r['id']}] ({r['pos']}) {r['gloss']}")
        
    out.append("\n### All Candidate Senses")
    for s in res["candidates"]:
        out.append(f"- [{s['id']}] ({s['pos']}) {s['gloss']}")
        
    top = res["ranked"][0]
    out.append("\n### Conclusion")
    out.append(f"**{top['prob']:.1%} is the highest match**, so the word **{res['target']}** most likely means:\n\n> [{top['id']}] ({top['pos']}) {top['gloss']}")
    
    if len(res["ranked"]) > 1:
        second = res["ranked"][1]
        margin = top["prob"] - second["prob"]
        if top["prob"] < 0.30 or margin < 0.05:
            out.append(f"\n_Low confidence: the next best sense is {second['prob']:.1%} ([{second['id']}] {second['gloss']}). Treat this as a suggestion, not a certain answer._")
            
    return "\n".join(out)

# Build the Gradio Application Object Structure
app = gr.Interface(
    fn=wsd_ui,
    inputs=[
        gr.Textbox(label="Kannada sentence", lines=3, placeholder="ಇಲ್ಲಿ ಕನ್ನಡ ವಾಕ್ಯವನ್ನು ನಮೂದಿಸಿ..."),
        gr.Textbox(label="Target word (as written in the sentence)", placeholder="ಪದವನ್ನು ನಮೂದಿಸಿ..."),
        gr.Textbox(label="Lemma (optional)", placeholder="ಧಾತು ರೂಪ (ಐಚ್ಛಿಕ)...")
    ],
    outputs=gr.Markdown(),
    title="Kannada Word Sense Disambiguation",
    description="Enter a sentence and the ambiguous word. Senses come from Kannada IndoWordNet.",
    flagging_mode="never",
)
app.launch(share=True, debug=True, prevent_thread_lock=True)


2026-11-20:14:03:29,874 INFO     [iwn.py:43] Loading kannada language synsets...


* Running on local URL:  http://127.0.0.1:7860


2026-11-20:14:03:55,406 INFO     [_client.py:1025] HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-11-20:14:03:55,735 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-11-20:14:03:56,526 INFO     [_client.py:1025] HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-11-20:14:03:56,717 INFO     [_client.py:1025] HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
2026-11-20:14:03:57,363 INFO     [_client.py:1025] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-11-20:14:03:59,493 INFO     [_client.py:1025] HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026-11-20:14:04:03,45 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-11-20:14:04:03,59 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-error-analytics "HTTP/1.1 200 OK"
